# T05 CP3 | Online Retail II: Cleaning, EDA, Bronze, Silver

**Team 5:** Boomikaa P, Chahatleen Kaur, Saba Gul, Vanshika Malviya, K. Shreya, Tushar Rathor.

Dataset: Chen (2012), UCI Online Retail II, CC BY 4.0, https://doi.org/10.24432/C5CG6D. Export both Excel sheets as CSV and upload them to the configured Unity Catalog Volume.

## 1. Set paths and table names

In [0]:
from pyspark.sql import functions as F

CATALOG = "retail_catalog"
SCHEMA = "team5_retail"
VOLUME = "retail_files"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
FILES = [
    f"{BASE}/online_retail_2009_2010.csv",
    f"{BASE}/online_retail_2010_2011.csv",
]
DB = f"{CATALOG}.{SCHEMA}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")


## 2. Load the source CSVs

In [0]:
parts = []
for file_path in FILES:
    part = (spark.read.option("header", True).option("mode", "PERMISSIVE")
            .csv(file_path)
            .withColumn("source_file", F.lit(file_path)))
    parts.append(part)
raw = parts[0].unionByName(parts[1], allowMissingColumns=True)
raw.printSchema()
raw.show(10, truncate=False)


## 3. Clean a working copy before EDA

In [0]:
renames = [("Invoice", "invoice"), ("StockCode", "stock_code"),
           ("Description", "description"), ("Quantity", "quantity"),
           ("InvoiceDate", "invoice_date"), ("Price", "price"),
           ("Customer ID", "customer_id"), ("Country", "country")]
cleaned = raw
for old_name, new_name in renames:
    cleaned = cleaned.withColumnRenamed(old_name, new_name)

cleaned = (cleaned
    .withColumn("invoice", F.trim("invoice"))
    .withColumn("stock_code", F.upper(F.trim("stock_code")))
    .withColumn("description", F.trim(F.regexp_replace("description", r"\s+", " ")))
    .withColumn("country", F.initcap(F.trim("country")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("customer_id", F.col("customer_id").cast("double").cast("long"))
    .withColumn("invoice_date", F.coalesce(
        F.try_to_timestamp("invoice_date", F.lit("M/d/yyyy H:mm")),
        F.to_timestamp("invoice_date")))
    .dropDuplicates()
    .withColumn("is_cancelled", F.upper(F.col("invoice")).startswith("C"))
    .withColumn("missing_customer_id", F.col("customer_id").isNull())
    .withColumn("zero_or_negative_price", F.col("price").isNull() | (F.col("price") <= 0))
    .withColumn("negative_quantity", F.col("quantity") < 0)
    .withColumn("missing_description", F.col("description").isNull() | (F.col("description") == "")))

print("Rows read:", raw.count())
print("Rows after exact duplicate removal:", cleaned.count())


## 4. EDA on the cleaned working copy

In [0]:
profile = cleaned.agg(
    F.count("*").alias("rows_after_cleaning"),
    F.countDistinct("invoice").alias("distinct_invoices"),
    F.countDistinct("stock_code").alias("distinct_stock_codes"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.countDistinct("country").alias("distinct_countries"),
    F.sum(F.col("missing_customer_id").cast("long")).alias("missing_customer_id_rows"),
    F.sum(F.col("missing_description").cast("long")).alias("blank_description_rows"),
    F.sum(F.col("is_cancelled").cast("long")).alias("cancellation_rows"),
    F.sum(F.col("negative_quantity").cast("long")).alias("negative_quantity_rows"),
    F.sum(F.col("zero_or_negative_price").cast("long")).alias("zero_or_negative_price_rows"),
    F.min("invoice_date").alias("first_invoice_date"),
    F.max("invoice_date").alias("last_invoice_date"),
    F.min("quantity").alias("minimum_quantity"), F.max("quantity").alias("maximum_quantity"),
    F.min("price").alias("minimum_price"), F.max("price").alias("maximum_price")
).first()
for name, value in profile.asDict().items():
    print(f"{name}: {value}")
if profile["rows_after_cleaning"]:
    missing_rate = 100 * profile["missing_customer_id_rows"] / profile["rows_after_cleaning"]
    print("missing customer ID rate (%):", round(missing_rate, 2))

print("Top countries by row count:")
cleaned.groupBy("country").count().orderBy(F.desc("count")).show(10, truncate=False)
print("Number summary:")
cleaned.select("quantity", "price").describe().show()


## 5. Bronze: save the unchanged source rows

In [0]:
bronze = raw.withColumnRenamed("Customer ID", "customer_id").withColumn("loaded_at", F.current_timestamp())
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.bronze_retail_raw")
print("Bronze raw rows:", raw.count())


## 6. Silver: save cleaned sales and cancellations separately

In [0]:
cleaned_cancellations = cleaned.filter(F.col("is_cancelled"))
cleaned_sales = cleaned.filter(~F.col("is_cancelled"))
cleaning_exceptions = cleaned_sales.filter(
    F.col("stock_code").isNull() | (F.col("stock_code") == "") |
    F.col("invoice_date").isNull() | F.col("quantity").isNull() | F.col("price").isNull())

cleaned_cancellations.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_cancellations")
cleaned_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_retail_sales")
cleaning_exceptions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{DB}.silver_quality_exceptions")


## 7. Save a data quality log

In [0]:
quality_rows = [
    ("raw_input_rows", str(raw.count())),
    ("rows_after_exact_duplicate_removal", str(cleaned.count())),
    ("cancellation_rows", str(cleaned_cancellations.count())),
    ("silver_sales_rows", str(cleaned_sales.count())),
    ("sales_missing_customer_id_rows", str(cleaned_sales.filter("missing_customer_id").count())),
    ("sales_negative_quantity_rows", str(cleaned_sales.filter("negative_quantity").count())),
    ("sales_zero_or_negative_price_rows", str(cleaned_sales.filter("zero_or_negative_price").count())),
    ("sales_quality_exception_rows", str(cleaning_exceptions.count())),
]
quality_log = spark.createDataFrame(quality_rows, ["check_name", "check_value"])
quality_log = quality_log.withColumn("logged_at", F.current_timestamp())
quality_log.write.format("delta").mode("overwrite").saveAsTable(f"{DB}.silver_quality_log")
quality_log.show(truncate=False)


## Notes

The provided workbook contains 1,067,371 rows across 1 Dec 2009 to 9 Dec 2011. A full local profile found missing customer IDs, repeated rows, cancellations, negative quantities, and zero/negative prices.